In [1]:

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:

import pandas as pd
path = "/content/drive/MyDrive/t60_reg_data.csv"
df = pd.read_csv(path)
df

,pat_id,age,gender,race_caucasian,race_black,race_asian,race_native_am,race_native_pacific,race_other,ethnicity,...,0_12h_heparin_line_flush_dose_total_12h,13_24h_heparin_line_flush_dose_total_12h,25_36h_heparin_line_flush_dose_total_12h,0_12h_urine_output_total_12h,13_24h_urine_output_total_12h,25_36h_urine_output_total_12h,0_12h_norepinephrine_equivalent_dose_total_12h,13_24h_norepinephrine_equivalent_dose_total_12h,25_36h_norepinephrine_equivalent_dose_total_12h,creatinine_ratio
0,Z1001193,55,1,2,2,2,2,2,2,2,...,0.0,0.0,0.0,1790.0,555.0,615,32.750000,25.013000,0.000000,0.804348
1,Z1002941,82,2,1,2,2,2,2,2,2,...,0.0,0.0,0.0,1345.0,690.0,385,4.000000,0.000000,0.000000,0.885714
2,Z1003573,73,2,2,2,2,2,2,2,2,...,0.0,0.0,0.0,910.0,580.0,350,19.058500,41.056333,12.539000,1.408163
3,Z1006132,58,1,1,2,2,2,2,2,2,...,0.0,0.0,0.0,2220.0,1405.0,1005,32.500000,96.692667,61.578000,0.953488
4,Z1006395,69,1,1,2,2,2,2,2,2,...,0.0,0.0,0.0,780.0,880.0,1015,83.013000,3.039000,0.000000,0.959732
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4529,Z997733,76,1,1,2,2,2,2,2,2,...,0.0,0.0,0.0,1330.0,460.0,440,109.526000,78.065000,41.513000,0.856164
4530,Z998224,57,1,2,2,2,2,2,2,2,...,0.0,0.0,0.0,2375.0,3375.0,950,156.277167,72.000000,71.333333,0.834081
4531,Z999344,67,1,2,2,2,2,2,2,2,...,0.0,0.0,0.0,1730.0,655.0,0,0.000000,0.000000,0.000000,0.940476
4532,Z999407,55,1,2,2,2,2,2,2,2,...,0.0,0.0,0.0,1195.0,720.0,1800,0.000000,0.000000,0.000000,0.712766


In [3]:

# --- AKI (t60) LLM score collection: structured outputs + batching ---
# pip install openai pydantic pandas numpy

import os
import json
import math
import time
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI


# -----------------------
# Load data + columns
# -----------------------
DATA_PATH = "/content/drive/MyDrive/t60_reg_data.csv"  # <-- change to your local path
ID_COLS = ["pat_id"]
TARGET_COL = "creatinine_ratio"   # regression target

df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
print("n_features =", len(FEATURE_COLS))


# -----------------------
# Prompt scaffolding
# -----------------------
DATASET_BACKGROUND = """
We have adult patients, 80 years and older, who underwent cardiac surgery.
For each patient, we have perioperative and ICU EMR-derived features:
demographics, comorbidities, vitals/hemodynamics, ventilator settings,
fluid balance, labs, procedures, and medication doses. Features are collected up until 36 hours post-surgery.
The modeling goal is sparse linear regression to predict the change in postoperative
serum creatinine after 60 hours. Specifically, we want to predict the ratio of postoperative
serum creatinine after 60 hours over the patient's baseline (pre-surgery) creatinine levels.
""".strip()

OUTCOME_NAME = "creatinine_ratio"

def infer_feature_type(series: pd.Series) -> str:
    """Light heuristic for the LLM (helps it reason about binary flags vs continuous)."""
    if pd.api.types.is_bool_dtype(series):
        return "binary"
    if pd.api.types.is_numeric_dtype(series):
        vals = series.dropna().unique()
        if len(vals) <= 3 and set(vals).issubset({0, 1}):
            return "binary"
        return "numeric"
    return "categorical_or_text"

def build_prompt_for_batch(feature_names: List[str]) -> str:
    # Include a tiny bit of metadata that is cheap but useful
    payload = []
    for i, name in enumerate(feature_names):
        ftype = infer_feature_type(df[name])
        payload.append({"id": i, "name": name, "type": ftype})

    features_json = json.dumps(payload, indent=2)

    prompt = f"""
You are a cardiothoracic ICU clinician, nephrologist, and biostatistician.

Background:
{DATASET_BACKGROUND}

Prediction target:
We are building a sparse linear regression model to predict {OUTCOME_NAME}, defined as the patient's
postoperative serum creatinine at Hour 60 divided by baseline (pre-surgery) creatinine.

Task:
For each feature, assign an integer importance score from 1 to 5 for its usefulness in predicting {OUTCOME_NAME}
in post-cardiac surgery patients age 80 and older.

Use this scoring strategy:
Judge each feature using a fixed clinical decomposition, then map the overall judgment to a final 1 to 5 score.

Clinical decomposition to use internally for every feature:
1. Renal Specificity:
   Does the feature directly reflect kidney function, kidney injury, renal perfusion, or a well-established AKI pathway?

2. Causal / Prognostic Proximity:
   Is the feature a direct biomarker, a strong physiological precursor, an established risk factor, or only an indirect proxy?

3. Temporal Proximity:
   How informative is this epoch for predicting the Hour 60 outcome?

4. Aggregation Quality:
   Does the chosen aggregation (_min, _max, _mean, _measured) capture the true pathology well?

5. Proxy Penalty:
   Is the feature mostly a marker of clinician behavior, monitoring intensity, or general illness severity rather than a renal mechanism?

Important scoring principles:
- Score absolute predictive usefulness for Hour 60 creatinine ratio, not just generic illness severity.
- Do not force score diversity. Many features may legitimately receive the same score.
- If uncertain between 2 adjacent scores, choose the lower score unless the feature is a direct kidney biomarker
  or a strong near-term indicator of renal hypoperfusion / impending AKI.
- Chronic baseline comorbidities are usually moderate predictors, not top-tier predictors, unless they are very tightly linked to renal reserve.
- Routine ICU maintenance variables should usually score low unless they directly encode renal risk, renal injury, or sustained hemodynamic instability.
- Medication variables should be scored based on renal relevance. Nephrotoxins, diuretics, vasopressors, and drugs strongly tied to perfusion
  or kidney injury may matter; generic supportive medications usually matter less.

Rules for Aggregated Vitals and Labs:
Many features end in _min, _max, _mean, and _measured and summarize a 12-hour post-surgery window.

Use these rules:
- _max / _min:
  Prefer extremes when acute insults are what matter clinically.
  Examples: hypotension, oliguria, peak lactate, peak creatinine, extreme vasopressor requirement.
  If the extreme state best captures the renal pathology, it can receive the full score range.

- _mean:
  Use a lower score than the corresponding extreme when the mean washes out short but clinically important insults.
  Only keep a high score if sustained exposure over the whole window is itself clinically meaningful.

- _measured:
  This is mostly a monitoring-intensity / clinician-suspicion proxy, not a physiological mechanism.
  Score _measured features as 1 or 2 by default.
  Only give a 3 if the measurement frequency itself is a meaningful signal of impending renal concern or severe instability.
  Do not assign 4 or 5 to _measured features.

Redundancy / collinearity rule:
We want sparse priors that reduce redundancy among closely related aggregated features.
When a family of variables differs only by aggregation, give the highest score to the aggregation that best captures the pathology,
and dampen the others if they are mostly redundant.

Rule for Temporal Proximity (Time Epochs):
Features may come from 3 consecutive 12-hour epochs: 0_12h, 13_24h, 25_36h.

Adjust scores by epoch:
- 0_12h (Immediate Postoperative Period):
  Derangements are common right after surgery, bypass, anesthesia, and resuscitation.
  Be conservative. Do not over-credit transient early abnormalities.

- 13_24h (Trajectory Phase):
  This window is more informative than 0_12h because it shows recovery vs deterioration,
  but it is still not the closest window to the outcome. Mild discount still applies.

- 25_36h (Leading Indicator Window):
  This is the closest physiological snapshot to the Hour 60 outcome.
  Full scoring range is allowed here.

What should score highest:
- Direct kidney function markers or strong near-term precursors of renal injury
- Features strongly linked to renal perfusion failure, sustained shock, or evolving AKI
- Features whose timing and aggregation preserve clinically meaningful signal

What should score lowest:
- Weakly related demographic or generic care-process variables
- Monitoring frequency variables
- Features with only vague or nonspecific association to renal injury

Final scoring rubric:
Use an integer importance score from 1 to 5.

Score 1:
Minimal or no meaningful relevance to Hour 60 creatinine ratio.
Mostly nonspecific, routine, or weak proxy information.

Score 2:
Plausible but weak or indirect association.
May reflect general acuity, treatment context, or mild renal relevance.

Score 3:
Established AKI risk factor or moderate predictor.
Clinically relevant, but not a direct or especially strong near-term indicator on its own.

Score 4:
Strong predictor with clear clinical linkage to subsequent AKI / worsening renal function.
Often a strong hemodynamic, biochemical, or treatment-related signal near the outcome window.

Score 5:
Direct renal biomarker or extremely strong near-term signal of evolving kidney injury or renal hypoperfusion.
Reserve this for features with especially tight mechanistic and prognostic linkage to Hour 60 creatinine ratio.

Input features (JSON list):
{features_json}

Return a JSON object with key "scores" containing a list of objects.
Each object must include:
- id (same as input id)
- name (copy exactly)
- importance (integer 1..5)
- reason (1 to 2 concise sentences)

In each reason:
- briefly identify what the feature likely represents clinically
- briefly justify the score using renal relevance, timing, aggregation quality, or proxy penalty
""".strip()

    return prompt

# -----------------------------------
# Structured output schema (Pydantic)
# -----------------------------------
class ScoreItem(BaseModel):
    id: int
    name: str
    importance: int = Field(ge=1, le=5)
    reason: str

class ScoreBatch(BaseModel):
    scores: List[ScoreItem]


# -----------------------
# OpenAI client + call
# -----------------------
# Set env var: export OPENAI_API_KEY="..."
client = OpenAI(api_key= "API KEY HERE")

def call_llm_for_batch(prompt: str, model: str = "gpt-5.2") -> ScoreBatch:
    """
    Uses Responses API structured parsing, so you don't need to manually parse JSON text.
    """
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": "You output only structured JSON that matches the schema."},
            {"role": "user", "content": prompt},
        ],
        text_format=ScoreBatch,
        temperature=0,
        max_output_tokens=2000,
    )
    return resp.output_parsed


def get_llm_importance_scores(
    feature_names: List[str],
    batch_size: Optional[int] = None,
    model: str = "gpt-5.2",
    sleep_s: float = 0.25,
    max_retries: int = 6,
    cache_path: str = "aki_llm_scores_reasoning.json",
) -> Dict[str, Dict]:
    """
    Returns:
      dict[feature_name] = {"importance": int(1..5), "reason": str}
    Caches incrementally so you can resume.
    """

    # default: ceil(sqrt(p)) like the paper’s heuristic
    if batch_size is None:
        batch_size = int(math.ceil(math.sqrt(len(feature_names))))

    # load cache if exists
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            results = json.load(f)
        print(f"[cache] loaded {len(results)} scores from {cache_path}")
    else:
        results = {}

    remaining = [f for f in feature_names if f not in results]
    print(f"Remaining features to score: {len(remaining)} (batch_size={batch_size})")

    for start in range(0, len(remaining), batch_size):
        batch = remaining[start : start + batch_size]
        prompt = build_prompt_for_batch(batch)

        # retry loop (simple exponential backoff)
        attempt = 0
        while True:
            try:
                parsed = call_llm_for_batch(prompt, model=model)
                # map back
                for item in parsed.scores:
                    results[item.name] = {
                        "importance": int(item.importance),
                        "reason": item.reason,
                    }
                # write cache
                with open(cache_path, "w") as f:
                    json.dump(results, f, indent=2)
                print(f"[ok] scored batch {start//batch_size + 1} / {math.ceil(len(remaining)/batch_size)}")
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {e}") from e
                backoff = (2 ** attempt) * 0.5
                print(f"[retry {attempt}] error={e} | sleeping {backoff:.1f}s")
                time.sleep(backoff)

        time.sleep(sleep_s)

    return results


# -----------------------
# Run + export to aki_weights.csv format used by your R code
# -----------------------
MODEL = "gpt-5.2"  # structured outputs supported on GPT-4o+ models :contentReference[oaicite:1]{index=1}

raw = get_llm_importance_scores(
    FEATURE_COLS,
    batch_size=16,          # defaults to ceil(sqrt(p))
    model=MODEL,
    cache_path="aki_llm_scores_reasoning.json",
)

# Save raw weights (1..5)
raw_df = pd.DataFrame({
    "value": list(raw.keys()),
    "importance": [raw[k]["importance"] for k in raw],
    "reason": [raw[k]["reason"] for k in raw],
})
raw_df.to_csv("aki_weights_reasoning.csv", index=False)

n_features = 1790
Remaining features to score: 1790 (batch_size=16)
[ok] scored batch 1 / 112
[ok] scored batch 2 / 112
[ok] scored batch 3 / 112
[ok] scored batch 4 / 112
[ok] scored batch 5 / 112
[ok] scored batch 6 / 112
[ok] scored batch 7 / 112
[ok] scored batch 8 / 112
[ok] scored batch 9 / 112
[ok] scored batch 10 / 112
[ok] scored batch 11 / 112
[ok] scored batch 12 / 112
[ok] scored batch 13 / 112
[ok] scored batch 14 / 112
[ok] scored batch 15 / 112
[ok] scored batch 16 / 112
[ok] scored batch 17 / 112
[ok] scored batch 18 / 112
[ok] scored batch 19 / 112
[ok] scored batch 20 / 112
[ok] scored batch 21 / 112
[ok] scored batch 22 / 112
[ok] scored batch 23 / 112
[ok] scored batch 24 / 112
[ok] scored batch 25 / 112
[ok] scored batch 26 / 112
[ok] scored batch 27 / 112
[ok] scored batch 28 / 112
[ok] scored batch 29 / 112
[ok] scored batch 30 / 112
[ok] scored batch 31 / 112
[ok] scored batch 32 / 112
[ok] scored batch 33 / 112
[ok] scored batch 34 / 112
[ok] scored batch 35 / 

In [4]:

from google.colab import files
files.download("aki_weights_reasoning.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:

df_new = pd.read_csv("aki_weights_reasoning.csv")
df_new["importance"].value_counts()

,count
importance,
1,761
2,670
3,266
4,78
5,15
